# Quickstart: aerial imaging with the F$_2$ excimer source

`highuvlith` simulates the optical-lithography pipeline for VUV/EUV/X-ray
sources. This notebook walks the **aerial-imaging** stage end to end for the
molecular-fluorine (F$_2$) excimer laser at **157.63 nm**, the workhorse VUV
source, imaged through a refractive CaF$_2$ projector.

We start from the one-line convenience API, then drop down to the full
`SimulationEngine` for control, sweep through focus, and finish with a
Bossung-style CD-versus-focus family.

### Physics context

The aerial image is the intensity pattern the mask projects into the resist
plane. `highuvlith` computes it with the **Hopkins partially-coherent** model:
the transmission cross-coefficient (TCC) is decomposed into a small number of
coherent **SOCS** kernels (sum-of-coherent-systems), and the image is the
incoherent sum of the kernel convolutions. Fewer kernels means a faster,
lower-rank approximation - we keep the count small here so everything runs in
seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import highuvlith as huv
from highuvlith import api

%matplotlib inline

print("highuvlith", huv.__version__ if hasattr(huv, "__version__") else "(dev)")

## 1. The one-liner: `simulate_line_space`

`api.simulate_line_space` builds the F$_2$ source, CaF$_2$ optics, a dense
line/space mask and the grid for you, then returns the aerial image plus
derived metrics. Here: **120 nm lines on a 300 nm pitch** at NA 0.75.

- **Image contrast** $= (I_{max}-I_{min})/(I_{max}+I_{min})$.
- **NILS** (normalized image log-slope) measures edge sharpness at the
  threshold; larger is better (a rule of thumb wants NILS $\gtrsim 2$).

In [ ]:
r = api.simulate_line_space(
    cd_nm=120, pitch_nm=300, na=0.75, grid_size=128, pixel_nm=5.0, max_kernels=12
)
print(f"image contrast : {r.contrast:.3f}")
print(f"NILS           : {r.nils:.2f}")

## 2. Full control: `SimulationEngine` + cross-section

For anything beyond the defaults you assemble the configs yourself and drive
the `SimulationEngine`. This exposes the raw 2-D intensity array and a
`cross_section(y_nm)` cut through the image, which we plot side by side.

In [ ]:
source = huv.SourceConfig.f2_laser()              # 157.63 nm, sigma_outer=0.7
optics = huv.OpticsConfig(numerical_aperture=0.75)
mask = huv.MaskConfig.line_space(cd_nm=120, pitch_nm=300)
grid = huv.GridConfig(size=128, pixel_nm=5.0)     # 640 nm field, ~2 pitches

engine = huv.SimulationEngine(source, optics, mask, grid=grid, max_kernels=12)
aerial = engine.compute_aerial_image(focus_nm=0.0)

intensity = np.asarray(aerial.intensity)          # (128, 128)
x, cut = np.asarray(aerial.cross_section(0.0))    # row 0 = x (nm), row 1 = I

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
extent = [x.min(), x.max(), x.min(), x.max()]
im = ax0.imshow(intensity, extent=extent, origin="lower", cmap="inferno")
ax0.set(title="Aerial image (F2, 157.63 nm)", xlabel="x (nm)", ylabel="y (nm)")
fig.colorbar(im, ax=ax0, fraction=0.046, label="intensity")

ax1.plot(x, cut, color="crimson")
ax1.set(title="Cross-section at y = 0", xlabel="x (nm)", ylabel="intensity")
ax1.grid(alpha=0.3)
fig.tight_layout()

## 3. Focus sweep: contrast versus defocus

`api.sweep_focus` batch-computes the aerial image across a range of defocus
values and returns the image contrast at each. The peak locates best focus;
the width of the peak previews the depth of focus.

In [ ]:
sweep = api.sweep_focus(
    cd_nm=120, pitch_nm=300, na=0.75,
    focus_min=-300, focus_max=300, focus_steps=13,
    grid_size=64, pixel_nm=10.0,
)
focuses = np.asarray(sweep["focuses"])
contrasts = np.asarray(sweep["contrasts"])

plt.figure(figsize=(6, 4))
plt.plot(focuses, contrasts, "o-", color="navy")
plt.axvline(sweep["best_focus_nm"], color="grey", ls="--",
            label=f"best focus = {sweep['best_focus_nm']:.0f} nm")
plt.xlabel("defocus (nm)")
plt.ylabel("image contrast")
plt.title("Contrast vs focus")
plt.legend()
plt.grid(alpha=0.3)
print(f"best focus = {sweep['best_focus_nm']:.0f} nm, "
      f"contrast = {sweep['best_contrast']:.3f}")

## 4. A Bossung-style CD-vs-focus family

The classic **Bossung plot** shows printed CD as a function of focus for a
family of exposure levels. `BatchSimulator.process_window` measures CD from
the aerial image at an iso-intensity threshold across a focus grid.

This is a **pure aerial-threshold** measurement (no resist develop model in
the batch path), so the exposure level enters only through that iso-intensity
threshold: a **lower** threshold clears more of the image and prints a
**narrower** line. We sweep the threshold to draw the Bossung fan, and read
off the depth of focus and exposure latitude for the middle curve.

In [ ]:
batch = huv.BatchSimulator(source, optics, mask, grid)
focus_grid = np.linspace(-150, 150, 7)

plt.figure(figsize=(6.5, 4))
for thresh in (0.22, 0.30, 0.38):
    pw = batch.process_window(
        [30.0], focus_grid.tolist(), cd_threshold=thresh, cd_target_nm=150.0
    )
    cd_vs_focus = np.asarray(pw.cd_matrix)[0]
    plt.plot(focus_grid, cd_vs_focus, "o-", label=f"threshold = {thresh:.2f}")

plt.xlabel("defocus (nm)")
plt.ylabel("printed CD (nm)")
plt.title("Bossung: CD vs focus at three iso-intensity thresholds")
plt.legend(title="lower = higher dose")
plt.grid(alpha=0.3)

# Depth of focus / exposure latitude for the middle threshold:
pw_mid = batch.process_window(
    [28.0, 30.0, 32.0], focus_grid.tolist(),
    cd_threshold=0.30, cd_target_nm=170.0, cd_tolerance_pct=10.0,
)
print(f"depth of focus    : {pw_mid.depth_of_focus():.0f} nm")
print(f"exposure latitude : {pw_mid.exposure_latitude():.1f} %")

### Where to go next

- Swap `huv.SourceConfig.f2_laser()` for any of the nine source families -
  see **`02_source_gallery.ipynb`**.
- Add a resist and develop model with `engine.compute_resist_profile(...)`.
- Push into the resist volume (LIGA, interference, grayscale) in
  **`03_deep_litho.ipynb`**.